# Synthetic Goban Data — Preview

Prototype for the synthetic data generator (`src/moku/synthetic.py`).
Generates realistic Go board images with perfect COCO annotations.

**Goal**: Verify visual quality before generating a full dataset.

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np

from moku.dataset import ID_TO_CATEGORY
from moku.synthetic import generate_synthetic_sample
from moku.viz import CATEGORY_COLORS, render_grid
from moku.grid import annotations_to_grid

## Single Sample Preview

Generate a single synthetic board and display it with bounding box annotations overlaid.

In [ ]:
def plot_synthetic_sample(image, annotation, show_labels=True):
    """Plot a synthetic sample with bounding boxes overlaid."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    # Left: annotated image
    ax = axes[0]
    ax.imshow(image)
    for bbox, cat_id in zip(annotation["objects"]["bbox"], annotation["objects"]["category"]):
        x, y, w, h = bbox
        color = CATEGORY_COLORS.get(cat_id, "red")
        lw = 3 if cat_id == 2 else 2
        rect = patches.Rectangle((x, y), w, h, linewidth=lw, edgecolor=color, facecolor="none")
        ax.add_patch(rect)
        if show_labels:
            label = ID_TO_CATEGORY[cat_id]
            ax.text(
                x, y - 2, label, fontsize=6, color="white",
                bbox=dict(boxstyle="round,pad=0.2", facecolor=color, alpha=0.8),
            )
    n_corners = sum(1 for c in annotation["objects"]["category"] if c == 2)
    n_black = sum(1 for c in annotation["objects"]["category"] if c == 0)
    n_white = sum(1 for c in annotation["objects"]["category"] if c == 1)
    ax.set_title(f"Synthetic sample \u2014 {n_corners} corners, {n_black} black, {n_white} white", fontsize=10)
    ax.axis("off")

    # Right: inferred grid
    grid = annotations_to_grid(annotation["objects"], board_size=19)
    render_grid(grid, ax=axes[1])
    n_mapped = int(np.count_nonzero(grid))
    axes[1].set_title(f"Inferred grid \u2014 {n_mapped}/{n_black + n_white} stones mapped", fontsize=10)

    plt.tight_layout()
    plt.show()


image, annotation = generate_synthetic_sample(board_size=19, image_size=640, n_stones=40)
plot_synthetic_sample(image, annotation)

## Grid of Samples

Generate a 3x3 grid to assess visual diversity: varying stone counts, board sizes, and perspective distortion.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(18, 18))

configs = [
    {"board_size": 19, "n_stones": 10, "perspective_strength": 0.0},
    {"board_size": 19, "n_stones": 50, "perspective_strength": 0.05},
    {"board_size": 19, "n_stones": 100, "perspective_strength": 0.1},
    {"board_size": 13, "n_stones": 8, "perspective_strength": 0.0},
    {"board_size": 13, "n_stones": 30, "perspective_strength": 0.06},
    {"board_size": 13, "n_stones": 50, "perspective_strength": 0.12},
    {"board_size": 9, "n_stones": 5, "perspective_strength": 0.0},
    {"board_size": 9, "n_stones": 20, "perspective_strength": 0.08},
    {"board_size": 9, "n_stones": 35, "perspective_strength": 0.15},
]

for ax, cfg in zip(axes.flat, configs):
    img, ann = generate_synthetic_sample(**cfg)
    ax.imshow(img)
    # Draw corner boxes
    for bbox, cat_id in zip(ann["objects"]["bbox"], ann["objects"]["category"]):
        if cat_id == 2:
            x, y, w, h = bbox
            rect = patches.Rectangle((x, y), w, h, linewidth=2, edgecolor=CATEGORY_COLORS[2], facecolor="none")
            ax.add_patch(rect)
    n_b = sum(1 for c in ann["objects"]["category"] if c == 0)
    n_w = sum(1 for c in ann["objects"]["category"] if c == 1)
    ax.set_title(f"{cfg['board_size']}\u00d7{cfg['board_size']}, {n_b}B+{n_w}W, persp={cfg['perspective_strength']}", fontsize=9)
    ax.axis("off")

plt.suptitle("Synthetic Goban Samples \u2014 Visual Quality Check", fontsize=14, y=0.98)
plt.tight_layout()
plt.show()

## Grid Inference Validation

Verify that the synthetic annotations produce correct grid mappings via the homography pipeline.
This confirms the bbox coordinates are accurate enough for downstream SGF conversion.

In [ ]:
# Generate 4 samples and show annotated image + inferred grid side by side
fig, axes = plt.subplots(4, 2, figsize=(16, 28))

for i in range(4):
    board_size = [9, 13, 19, 19][i]
    n_stones = [15, 25, 40, 80][i]
    persp = [0.0, 0.06, 0.08, 0.12][i]

    img, ann = generate_synthetic_sample(
        board_size=board_size, image_size=640,
        n_stones=n_stones, perspective_strength=persp,
    )

    # Left: annotated image
    ax_img = axes[i, 0]
    ax_img.imshow(img)
    for bbox, cat_id in zip(ann["objects"]["bbox"], ann["objects"]["category"]):
        x, y, w, h = bbox
        color = CATEGORY_COLORS.get(cat_id, "red")
        lw = 3 if cat_id == 2 else 1.5
        rect = patches.Rectangle((x, y), w, h, linewidth=lw, edgecolor=color, facecolor="none")
        ax_img.add_patch(rect)
    n_b = sum(1 for c in ann["objects"]["category"] if c == 0)
    n_w = sum(1 for c in ann["objects"]["category"] if c == 1)
    ax_img.set_title(f"{board_size}\u00d7{board_size}, {n_b}B+{n_w}W, perspective={persp}", fontsize=10)
    ax_img.axis("off")

    # Right: inferred grid
    grid = annotations_to_grid(ann["objects"], board_size=board_size)
    render_grid(grid, ax=axes[i, 1])
    n_mapped = int(np.count_nonzero(grid))
    axes[i, 1].set_title(f"Inferred grid \u2014 {n_mapped}/{n_b + n_w} stones mapped", fontsize=10)

plt.suptitle("Synthetic Samples \u2192 Grid Inference Pipeline Check", fontsize=14, y=0.99)
plt.tight_layout()
plt.show()